In [ ]:
# ============================================================
# MiniCheck Evaluation from qa_data.json
# ============================================================

!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"
!pip -q install pandas scikit-learn tqdm

import os
import json
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from minicheck.minicheck import MiniCheck

# ============================================================
# CONFIG
# ============================================================

JSON_FILE = "/kaggle/input/datasets/belalheshamfathy/ffffffffff/qa_data_answer_focused_summarized.json"

OUTPUT_DIR = "/kaggle/working/minicheck_outputs_summ"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# للتجربة السريعة
MAX_SAMPLES = 6000

# للرن الكامل
# MAX_SAMPLES = None

MINICHECK_MODEL = "flan-t5-large"

BATCH_SIZE = 8

# ============================================================
# LOAD JSON
# ============================================================

with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for idx, item in enumerate(data):

    knowledge = item["knowledge"]
    question = item["question"]

    right_answer = item["right_answer"]
    hallucinated_answer = item["hallucinated_answer"]

    rows.append({
        "id": idx,
        "knowledge": knowledge,
        "question": question,
        "answer": right_answer,
        "label": 1
    })

    rows.append({
        "id": idx,
        "knowledge": knowledge,
        "question": question,
        "answer": hallucinated_answer,
        "label": 0
    })

df = pd.DataFrame(rows)

print("Total samples:", len(df))
print(df["label"].value_counts())

# ============================================================
# OPTIONAL SUBSET
# ============================================================

if MAX_SAMPLES is not None:

    df0 = df[df.label == 0].sample(
        MAX_SAMPLES // 2,
        random_state=42
    )

    df1 = df[df.label == 1].sample(
        MAX_SAMPLES // 2,
        random_state=42
    )

    df = pd.concat([df0, df1]).sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

print("Evaluation samples:", len(df))

# ============================================================
# CLAIM FORMAT
# ============================================================

def build_claim(question, answer):
    return f"Question: {question}\nAnswer: {answer}"

claims = [
    build_claim(q, a)
    for q, a in zip(df.question, df.answer)
]

docs = df.knowledge.astype(str).tolist()

# ============================================================
# LOAD MINICHECK
# ============================================================

print("Loading MiniCheck...")

scorer = MiniCheck(
    model_name=MINICHECK_MODEL,
    cache_dir="./ckpts"
)

print("Loaded.")

# ============================================================
# INFERENCE
# ============================================================

preds = []
probs = []

for i in tqdm(
    range(0, len(df), BATCH_SIZE),
    desc="MiniCheck"
):

    batch_docs = docs[i:i+BATCH_SIZE]
    batch_claims = claims[i:i+BATCH_SIZE]

    pred_label, raw_prob, _, _ = scorer.score(
        docs=batch_docs,
        claims=batch_claims
    )

    preds.extend(pred_label)
    probs.extend(raw_prob)

df["prediction"] = preds
df["probability"] = probs

# ============================================================
# METRICS
# ============================================================

acc = accuracy_score(df.label, df.prediction)

precision, recall, f1, _ = precision_recall_fscore_support(
    df.label,
    df.prediction,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    df.label,
    df.prediction,
    labels=[0,1]
)

print("\n===== RESULTS =====")
print("Accuracy :", round(acc,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("Macro F1 :", round(f1,4))

print("\nConfusion Matrix")
print(cm)

print("\nClassification Report")
print(
    classification_report(
        df.label,
        df.prediction,
        target_names=[
            "Hallucinated",
            "Supported"
        ],
        zero_division=0
    )
)

# ============================================================
# SAVE
# ============================================================

pred_file = os.path.join(
    OUTPUT_DIR,
    "minicheck_predictions.csv"
)

df.to_csv(pred_file, index=False)

print("\nSaved:")
print(pred_file)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Total samples: 20000
label
1    10000
0    10000
Name: count, dtype: int64
Evaluation samples: 6000
Loading MiniCheck...


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded.


MiniCheck:   0%|          | 0/750 [00:00<?, ?it/s]


Evaluating: 100%|██████████| 8/8 [00:00<00:00, 12.18it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.82it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.82it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 14.15it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.50it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.38it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.36it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.08it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.72it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.40it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.15it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.51it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 12.79it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.74it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.34it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.47it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 13.89it/

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

CSV_PATH = "/kaggle/working/minicheck_outputs_summ/minicheck_predictions.csv"

df = pd.read_csv(CSV_PATH)

print(df.columns.tolist())
print("Total:", len(df))
print("Labels:", df["label"].value_counts().to_dict())

# لو اسم عمود التوقع prediction
pred_col = "prediction"

acc = accuracy_score(df["label"], df[pred_col])
p, r, f1, _ = precision_recall_fscore_support(
    df["label"], df[pred_col], average="macro", zero_division=0
)

cm = confusion_matrix(df["label"], df[pred_col], labels=[0, 1])

print("\n===== MiniCheck Results =====")
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {p:.4f}")
print(f"Macro Recall    : {r:.4f}")
print(f"Macro F1        : {f1:.4f}")

print("\nConfusion Matrix [0,1]:")
print(cm)

print("\nClassification Report:")
print(classification_report(df["label"], df[pred_col], target_names=["Hallucinated", "Supported"], zero_division=0))

['id', 'knowledge', 'question', 'answer', 'label', 'prediction', 'probability']
Total: 6000
Labels: {0: 3000, 1: 3000}

===== MiniCheck Results =====
Accuracy        : 0.7400
Macro Precision : 0.7487
Macro Recall    : 0.7400
Macro F1        : 0.7377

Confusion Matrix [0,1]:
[[2500  500]
 [1060 1940]]

Classification Report:
              precision    recall  f1-score   support

Hallucinated       0.70      0.83      0.76      3000
   Supported       0.80      0.65      0.71      3000

    accuracy                           0.74      6000
   macro avg       0.75      0.74      0.74      6000
weighted avg       0.75      0.74      0.74      6000

